# 모델 예측 시각화

**전제조건**: `python -m python.train_pipeline` 실행 후 사용

## 시각화 목록
1. 기준기온 회귀 — ASOS 실측 vs 모델 예측 (연도/ONI 효과)
2. ONI 변화에 따른 기온 변동 (월별)
3. 구별 기온 오프셋 히트맵 (25구 × 12월)
4. 연간 그래프 — ONI 고정, 연도별 구별 총소비량
5. 공급량 & 예비율 — 연도별 추세
6. 과거 vs 시뮬레이션 비교 (특정 연도, ONI 변경)

In [ ]:
import sys
sys.path.insert(0, "../..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib

# 한글 폰트
matplotlib.rc("font", family="AppleGothic")
matplotlib.rc("axes", unicode_minus=False)
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ── 데이터 & 모델 로드 ──────────────────────────────────────────────────────
from python.loader.asos_daily import load_hourly, to_daily, to_monthly
from python.loader.oni_loader import load_oni
from python.loader.supply_loader import load_monthly as load_supply
from python.loader.kepco_loader import load_consumption
from python.train.temp_trend import load_model as load_temp_model, predict as pred_temp
from python.train.supply_regression import load_model as load_supply_model, predict as pred_supply
from python.preprocess.gu_offset import load_gu_params, predict_gu_temp

print("모델 로드 중...")
temp_model   = load_temp_model()
supply_model = load_supply_model()
monthly_offset, ws_clim = load_gu_params()

print("데이터 로드 중...")
daily        = to_daily(load_hourly())
monthly_asos = to_monthly(daily)
oni_df       = load_oni()
supply_df    = load_supply()
consumption  = load_consumption()
temp_gu      = pd.read_csv("../../data/output/temperature_gu_monthly.csv", encoding="utf-8-sig")

print(f"  ASOS monthly: {monthly_asos.shape} | ONI: {oni_df.shape}")
print(f"  공급량: {supply_df.shape} | 소비량: {consumption.shape}")
print(f"  구별기온: {temp_gu.shape}")
print("완료")

---
## 1. 기준기온 회귀 — 실측 vs 예측

In [ ]:
# 실측과 예측을 merge하여 비교
df_comp = monthly_asos.merge(oni_df, on=["year", "month"], how="inner").copy()
df_comp["ta_pred"] = df_comp.apply(
    lambda r: pred_temp(int(r.year), int(r.month), float(r.oni), temp_model), axis=1
)

# 여름(7~8월)만 추출
summer = df_comp[df_comp["month"].isin([7, 8])].copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 월 산점도
ax = axes[0]
ax.scatter(df_comp["ta_mean"], df_comp["ta_pred"], alpha=0.4, s=15, color="steelblue")
lim = [df_comp[["ta_mean","ta_pred"]].min().min()-1, df_comp[["ta_mean","ta_pred"]].max().max()+1]
ax.plot(lim, lim, "r--", lw=1)
ax.set_xlabel("실측 기온 (℃)"); ax.set_ylabel("예측 기온 (℃)")
ax.set_title("ASOS 기온 실측 vs 회귀 예측 (전체 월)")
corr = df_comp[["ta_mean","ta_pred"]].corr().iloc[0,1]
ax.text(0.05, 0.92, f"r = {corr:.3f}", transform=ax.transAxes, fontsize=11)

# 여름 연도별 시계열
ax = axes[1]
for month, label, color in [(7, "7월", "coral"), (8, "8월", "steelblue")]:
    sub = df_comp[df_comp["month"]==month].sort_values("year")
    ax.plot(sub["year"], sub["ta_mean"], "o--", color=color, label=f"{label} 실측", markersize=4)
    ax.plot(sub["year"], sub["ta_pred"], "-",  color=color, label=f"{label} 예측", alpha=0.7)
ax.set_xlabel("연도"); ax.set_ylabel("기온 (℃)")
ax.set_title("여름철 기온 추세 (실측 vs 예측)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 2. ONI 변화에 따른 기온 변동

In [ ]:
# 고정 연도에서 ONI를 -2.0 ~ +2.0 변화시킬 때 기온 변동
oni_range  = np.arange(-2.0, 2.1, 0.1)
years_show = [2015, 2020, 2025, 2030]
months_show = [1, 4, 7, 10]   # 겨울/봄/여름/가을
month_labels = {1:"1월(겨울)", 4:"4월(봄)", 7:"7월(여름)", 10:"10월(가을)"}

fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)

for ax, m in zip(axes.flat, months_show):
    for yr, color in zip(years_show, ["navy","steelblue","coral","firebrick"]):
        temps = [pred_temp(yr, m, o, temp_model) for o in oni_range]
        ax.plot(oni_range, temps, label=f"{yr}년", color=color)
    ax.axvline(0, color="gray", lw=0.8, ls="--")
    ax.set_title(f"{month_labels[m]}")
    ax.set_xlabel("ONI")
    ax.set_ylabel("예측 기온 (℃)")
    ax.legend(fontsize=8)

fig.suptitle("ONI 변화 → 기온 변동 (연도·계절별)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. 구별 기온 오프셋 히트맵

In [ ]:
# monthly_offset {(gu, month): offset} → 히트맵
gu_list = sorted(set(k[0] for k in monthly_offset))
offset_matrix = pd.DataFrame(
    index=gu_list,
    columns=range(1, 13),
    data=[[monthly_offset.get((gu, m), 0.0) for m in range(1, 13)] for gu in gu_list],
    dtype=float,
)

fig, ax = plt.subplots(figsize=(13, 9))
im = ax.imshow(offset_matrix.values, cmap="RdYlBu_r", aspect="auto")
plt.colorbar(im, ax=ax, label="오프셋 (℃)")

ax.set_xticks(range(12))
ax.set_xticklabels([f"{m}월" for m in range(1, 13)])
ax.set_yticks(range(len(gu_list)))
ax.set_yticklabels(gu_list)
ax.set_title("구별 월별 기온 오프셋 (S-DoT − ASOS, ℃)\n양수: ASOS 대비 더 더움 (열섬)")

# 값 표시
for i, gu in enumerate(gu_list):
    for j, m in enumerate(range(1, 13)):
        v = offset_matrix.iloc[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=6.5,
                color="white" if abs(v) > 1.2 else "black")

plt.tight_layout()
plt.show()

---
## 4. 연간 그래프 — ONI 고정, 연도별 구별 총소비량

In [ ]:
from python.train.consumption_xgb import load_model as load_cons_model, predict_all_districts

cons_model, cons_enc = load_cons_model()
USAGE_TYPES = ["가로등", "교육용", "농사용", "산업용", "심야", "일반용", "주택용"]

# ONI=0.0 고정, 연도별 8월 구별 총소비량
years_plot = list(range(2010, 2031, 5))  # 2010, 2015, 2020, 2025, 2030
oni_fixed  = 0.0
month_fixed = 8

records = []
for yr in years_plot:
    ta_asos = pred_temp(yr, month_fixed, oni_fixed, temp_model)
    gu_temps = predict_gu_temp(ta_asos, month_fixed, monthly_offset, multiplier=1.0)
    df_c = predict_all_districts(
        year=yr, month=month_fixed, oni=oni_fixed,
        gu_temps=gu_temps, usage_types=USAGE_TYPES,
        model=cons_model, encoders=cons_enc,
    )
    total = df_c.groupby("district")["consumption_mwh"].sum().reset_index()
    total["year"] = yr
    records.append(total)

df_plot = pd.concat(records, ignore_index=True)

# 상위 10개 구만 표시
top10 = df_plot[df_plot["year"]==2025].nlargest(10, "consumption_mwh")["district"].tolist()
df_top = df_plot[df_plot["district"].isin(top10)]

fig, ax = plt.subplots(figsize=(13, 6))
for gu, grp in df_top.groupby("district"):
    ax.plot(grp["year"], grp["consumption_mwh"], marker="o", label=gu)
ax.set_xlabel("연도")
ax.set_ylabel("총소비량 (MWh)")
ax.set_title(f"구별 총 전력소비량 예측 (8월, ONI={oni_fixed}) — 상위 10구")
ax.legend(fontsize=8, loc="upper left")
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

---
## 5. 공급량 & 예비율 — 연도별 추세

In [ ]:
import calendar as cal

# 8월 기준, ONI=-1/0/+1 시나리오
years_all = list(range(2005, 2031))
scenarios = {"La Niña (ONI=-1.0)": -1.0, "평년 (ONI=0.0)": 0.0, "El Niño (ONI=+1.0)": 1.0}
month_s = 8

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["steelblue", "gray", "coral"]

for (label, oni_val), color in zip(scenarios.items(), colors):
    supply_list, reserve_list = [], []
    for yr in years_all:
        ta = pred_temp(yr, month_s, oni_val, temp_model)
        days = cal.monthrange(yr, month_s)[1]
        cdd = max(0.0, ta - 24.0) * days
        hdd = max(0.0, 18.0 - ta) * days
        r = pred_supply(yr, month_s, oni_val, cdd, hdd, supply_model)
        supply_list.append(r["supply_mw"])
        reserve_list.append(r["reserve_rate"])
    axes[0].plot(years_all, supply_list, label=label, color=color)
    axes[1].plot(years_all, reserve_list, label=label, color=color)

# 실측값 overlay
actual_aug = supply_df[supply_df["month"]==8].sort_values("year")
axes[0].scatter(actual_aug["year"], actual_aug["supply_mw_mean"],
                color="black", s=20, zorder=5, label="실측")
axes[1].scatter(actual_aug["year"], actual_aug["reserve_rate_mean"],
                color="black", s=20, zorder=5, label="실측")

axes[0].set_title("8월 공급량 예측 (MW)"); axes[0].set_xlabel("연도")
axes[0].set_ylabel("공급능력 (MW)"); axes[0].legend(fontsize=8)
axes[0].yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))

axes[1].set_title("8월 공급예비율 예측 (%)"); axes[1].set_xlabel("연도")
axes[1].set_ylabel("예비율 (%)")
axes[1].axhline(10, color="red", ls="--", lw=1, label="위험 기준 10%")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 6. 과거 vs 시뮬레이션 비교

특정 연도의 ONI를 실측값에서 변경했을 때 전력소비량·기온이 어떻게 달라지는지

In [ ]:
# 2023년 8월 — 실측 ONI vs 변경 ONI
year_cmp, month_cmp = 2023, 8

oni_actual_val = oni_df[(oni_df["year"]==year_cmp) & (oni_df["month"]==month_cmp)]["oni"].values[0]
oni_sim_vals   = [-2.0, -1.0, 0.0, 1.0, 2.0]

rows = []
for oni_v in oni_sim_vals:
    ta_a = pred_temp(year_cmp, month_cmp, oni_v, temp_model)
    gu_t = predict_gu_temp(ta_a, month_cmp, monthly_offset, multiplier=1.0)
    df_c = predict_all_districts(
        year=year_cmp, month=month_cmp, oni=oni_v,
        gu_temps=gu_t, usage_types=USAGE_TYPES,
        model=cons_model, encoders=cons_enc,
    )
    total_seoul = df_c["consumption_mwh"].sum()
    rows.append({"oni": oni_v, "asos_temp": round(ta_a,2), "total_mwh": total_seoul,
                 "is_actual": abs(oni_v - oni_actual_val) < 0.05})

df_cmp = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, ylabel in zip(axes,
                            ["asos_temp", "total_mwh"],
                            ["ASOS 예측 기온 (℃)", "서울 전체 소비량 (MWh)"]):
    bars = ax.bar(df_cmp["oni"].astype(str), df_cmp[col],
                  color=["coral" if r else "steelblue" for r in df_cmp["is_actual"]])
    ax.set_xlabel("ONI"); ax.set_ylabel(ylabel)
    ax.set_title(f"{year_cmp}년 {month_cmp}월 — ONI 시나리오별 비교\n(붉은 막대: 실측 ONI={oni_actual_val:.2f})")
    if col == "total_mwh":
        ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))

plt.tight_layout()
plt.show()

print(df_cmp.to_string(index=False))

---
## 7. 월별 12개월 연간 그래프 (API /predict/annual 동치)

In [ ]:
# 연도 선택, ONI 고정
YEAR = 2025
ONI  = 0.5

months_all = list(range(1, 13))
asos_temps, supply_mws, reserve_rates, seoul_totals = [], [], [], []

for m in months_all:
    ta = pred_temp(YEAR, m, ONI, temp_model)
    days = cal.monthrange(YEAR, m)[1]
    cdd = max(0.0, ta - 24.0) * days
    hdd = max(0.0, 18.0 - ta) * days
    sp  = pred_supply(YEAR, m, ONI, cdd, hdd, supply_model)
    gu_t = predict_gu_temp(ta, m, monthly_offset, multiplier=1.0)
    df_c = predict_all_districts(
        year=YEAR, month=m, oni=ONI,
        gu_temps=gu_t, usage_types=USAGE_TYPES,
        model=cons_model, encoders=cons_enc,
    )
    asos_temps.append(ta)
    supply_mws.append(sp["supply_mw"])
    reserve_rates.append(sp["reserve_rate"])
    seoul_totals.append(df_c["consumption_mwh"].sum())

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
x = months_all
xlabels = [f"{m}월" for m in months_all]

axes[0,0].plot(x, asos_temps, "o-", color="coral")
axes[0,0].set_title(f"{YEAR}년 월별 ASOS 기온 (ONI={ONI})")
axes[0,0].set_ylabel("기온 (℃)")

axes[0,1].bar(x, supply_mws, color="steelblue")
axes[0,1].set_title(f"{YEAR}년 월별 공급량 (MW)")
axes[0,1].set_ylabel("공급능력 (MW)")
axes[0,1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f"{v:,.0f}"))

axes[1,0].plot(x, reserve_rates, "s-", color="green")
axes[1,0].axhline(10, color="red", ls="--", lw=1)
axes[1,0].set_title(f"{YEAR}년 월별 공급예비율 (%)")
axes[1,0].set_ylabel("예비율 (%)")

axes[1,1].bar(x, seoul_totals, color="mediumpurple")
axes[1,1].set_title(f"{YEAR}년 월별 서울 전체 소비량 (MWh)")
axes[1,1].set_ylabel("소비량 (MWh)")
axes[1,1].yaxis.set_major_formatter(ticker.FuncFormatter(lambda v,_: f"{v:,.0f}"))

for ax in axes.flat:
    ax.set_xticks(x); ax.set_xticklabels(xlabels)

plt.tight_layout()
plt.show()